In [1]:
print(123)

123


Q1: "I just discovered the course. Can I still join it?"
Q2: "I just found out about the program. Can I still enroll?"

These two are close - they mean the same thing.

Q3: "How do I run Docker on Windows?"

This one is far away from Q1 and Q2.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1

array([ 2.13904120e-02, -7.39799663e-02,  1.42068556e-03,  2.13816613e-02,
        2.45113149e-02,  3.15582752e-02, -1.10839702e-01, -1.05017498e-01,
       -6.18258938e-02, -6.42311852e-03,  3.72399064e-03,  9.06393602e-02,
       -9.49941017e-03,  6.53976873e-02,  1.10946558e-02, -2.10097339e-02,
       -3.35125513e-02, -4.31677327e-02,  9.96348727e-03,  1.41969863e-02,
       -6.40414879e-02, -7.04182126e-03, -7.91188031e-02,  5.80030754e-02,
        1.30212074e-03,  4.19733534e-03,  5.70979156e-02,  6.39447570e-02,
        2.49902904e-02, -3.95876691e-02, -3.79506797e-02,  2.70394552e-02,
        1.79423206e-02,  1.72272157e-02,  3.43311131e-02,  9.29055270e-03,
        5.86054735e-02, -4.97789569e-02, -5.05366083e-03,  4.34328541e-02,
       -1.56622957e-02, -2.97534503e-02, -5.13325352e-03,  5.13414629e-02,
        6.16060290e-03,  6.86980635e-02, -1.29505778e-02, -5.61938696e-02,
       -1.08265011e-02,  5.96683845e-02,  5.29939681e-02, -3.42755206e-02,
       -4.15274203e-02, -

In [5]:
v1.shape

(384,)

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.32332397)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [9]:
v2.dot(dv)

np.float32(0.019730438)

In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [13]:
len(texts)

1368

In [14]:
from tqdm.auto import tqdm

In [15]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [17]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [18]:
scores[:10]

[np.float32(0.48740578),
 np.float32(0.20991932),
 np.float32(0.76294106),
 np.float32(0.4437854),
 np.float32(0.2608399),
 np.float32(0.48665166),
 np.float32(0.30061564),
 np.float32(0.56009996),
 np.float32(0.4596049),
 np.float32(0.2562817)]

In [19]:
import numpy as np
X = np.array(vectors)
X

array([[-0.02670622, -0.12245756,  0.01594418, ..., -0.00230645,
        -0.11218391, -0.02365551],
       [-0.01099553, -0.11074745, -0.02536936, ...,  0.09022227,
        -0.02697366,  0.01975676],
       [-0.08896549, -0.06128184,  0.00775605, ...,  0.0405971 ,
         0.0047928 , -0.0274594 ],
       ...,
       [-0.03652925,  0.01415433, -0.06838643, ...,  0.04316785,
         0.08105534, -0.02148626],
       [-0.13091588, -0.06990597, -0.00931881, ..., -0.00044341,
        -0.01285729,  0.01426927],
       [-0.07984771,  0.01926989,  0.02544983, ..., -0.03368025,
        -0.01884021,  0.05837058]], shape=(1368, 384), dtype=float32)

In [20]:
scores = X.dot(v1)

In [21]:
scores

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759792, -0.03037044], shape=(1368,), dtype=float32)

In [22]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [23]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 643, 925, 538,   7])

In [29]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [35]:
#or instead of writing those extra boxes above to get the scores to display from highest to lowest, you can do the below:

top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 643, 925, 538,   7])

In [ ]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [37]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [38]:
vindex.search(v1)

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'doc_id': '2d8b16c2a0'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'quest

In [41]:
vindex.search(v1, num_results=5, filter_dict={"course": "llm-zoomcamp"})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an